In [0]:
CREATE OR REPLACE TABLE workspace.default.silver_products as 
WITH parsed as (
    SELECT
         trim(name)                              as product_name,
         nullif(trim(Brand), '')                 as brand,
         cast(price as double)                   as mrp,
         cast(discountedPrice as double)         as selling_price,
         trim(category)                          as category,
         trim(Subcategory)                       as sub_category,
         trim(quantity)                          as quantity_raw,
            TRY_CAST(NULLIF(regexp_extract(trim(Quantity), '^([0-9.]+) *([A-Za-z]+)$', 1), '') AS DOUBLE) AS qty_num,
    lower(regexp_extract(trim(Quantity), '^([0-9.]+) *([A-Za-z]+)$', 2)) AS qty_unit
  FROM workspace.default.bronze_products
  WHERE Name IS NOT NULL AND Price > 0
)

SELECT
  product_name, brand, mrp, selling_price, category, sub_category, quantity_raw,
  ROUND((mrp - selling_price) / mrp * 100, 1) AS discount_pct,
  CASE WHEN qty_unit = 'kg' THEN qty_num * 1000
       WHEN qty_unit = 'gm' THEN qty_num END   AS weight_g,
  CASE WHEN qty_unit IN ('l', 'litres') THEN qty_num * 1000
       WHEN qty_unit = 'ml' THEN qty_num END   AS volume_ml
FROM parsed;


select count(*) as row_count,
       count_if(weight_g is not null) as with_weight,
       count_if(volume_ml is not null) as with_volume,
       count_if(discount_pct < 0) as negative_discounts
from workspace.default.silver_products;